# MSWEP Indonesia DJF EOF/PCA

EOF/PCA of DJF rainfall anomalies over Indonesia (`lat -15..10`, `lon 90..155`) using MSWEP monthly precipitation.

- DJF definition: `D(y-1), J(y), F(y)`
- Anomaly baseline: `1991-2020`
- EOF method: covariance-based PCA (no latitude weighting, no pointwise standardization)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter

# -------------------------
# User parameters
# -------------------------
DATA_PATH = Path('/Users/rizzie/Academic/9_TugasAkhir/data/mswep/precip-mon-mswep-1979-2025.nc')
NINO34_PATH = Path('/Users/rizzie/Academic/9_TugasAkhir/data/index/nino34.anom.csv')
LAT_MIN, LAT_MAX = -15.0, 10.0
LON_MIN, LON_MAX = 90.0, 155.0
CLIM_START, CLIM_END = 1991, 2020
EOF_NMODES = 4
N_PLOT_MODES = 3

OUT_BASE = Path('/Users/rizzie/Academic/9_TugasAkhir/notebook/non-stationarity/output_eof')
OUT_PNG = OUT_BASE / 'png'
OUT_TABLES = OUT_BASE / 'tables'
OUT_PNG.mkdir(parents=True, exist_ok=True)
OUT_TABLES.mkdir(parents=True, exist_ok=True)

print(f'DATA_PATH: {DATA_PATH}')
print(f'NINO34_PATH: {NINO34_PATH}')
print(f'Output PNG: {OUT_PNG}')
print(f'Output tables: {OUT_TABLES}')


In [ ]:
ds = xr.open_dataset(DATA_PATH)
if 'precipitation' not in ds.data_vars:
    raise KeyError("Variable 'precipitation' not found in dataset")

pr = ds['precipitation']

# Handle latitude orientation safely.
lat0 = float(pr.lat.values[0])
lat1 = float(pr.lat.values[-1])
lat_slice = slice(LAT_MAX, LAT_MIN) if lat0 > lat1 else slice(LAT_MIN, LAT_MAX)

pr_id = pr.sel(lat=lat_slice, lon=slice(LON_MIN, LON_MAX)).sortby('lat')

print('Subset shape (time, lat, lon):', pr_id.shape)
print('Lat range:', float(pr_id.lat.min().values), 'to', float(pr_id.lat.max().values))
print('Lon range:', float(pr_id.lon.min().values), 'to', float(pr_id.lon.max().values))
print('Time range:', pd.Timestamp(pr_id.time.min().values), 'to', pd.Timestamp(pr_id.time.max().values))


In [ ]:
def build_complete_son(da: xr.DataArray) -> xr.DataArray:
    """Build SON seasonal means with SON year label = September year."""
    month = da['time'].dt.month
    year = da['time'].dt.year

    is_son = month.isin([9, 10, 11])
    da_son = da.sel(time=is_son)

    son_year = da_son['time'].dt.year
    da_son = da_son.assign_coords(son_year=('time', son_year.data))

    month_count = da_son['time'].groupby('son_year').count()
    full_years = month_count['son_year'].where(month_count == 3, drop=True).values.astype(int)

    son = da_son.groupby('son_year').mean('time').sel(son_year=full_years)
    son = son.rename({'son_year': 'year'}).sortby('year')
    return son

son = build_complete_son(pr_id)
years = son['year'].values.astype(int)

print('SON years:', years[0], 'to', years[-1], f'(n={len(years)})')

# Time coverage check for expected first and last complete SON.
monthly_times = pd.DatetimeIndex(pr_id.time.values)
for t_expected in [pd.Timestamp('1979-09-01'), pd.Timestamp('2024-11-01')]:
    if t_expected not in monthly_times:
        raise AssertionError(f'Missing required month for full SON span: {t_expected}')

expected_years = np.arange(1979, 2025)
if not np.array_equal(years, expected_years):
    raise AssertionError(f'SON years mismatch. Got {years[0]}..{years[-1]} (n={len(years)}), expected 1979..2025 (n=47)')

# SON rule spot-check for year 2000: S(2000), O(2000), N(2000).
test_months = pd.to_datetime(['2000-09-01', '2000-10-01', '2000-11-01'])
for tm in test_months:
    if tm not in monthly_times:
        raise AssertionError(f'Missing month for SON(2000) check: {tm}')

manual_2000 = pr_id.sel(time=test_months).mean('time')
if not np.allclose(manual_2000.values, son.sel(year=2000).values, equal_nan=True):
    raise AssertionError('SON(2000) mean mismatch against manual S(2000)-O(2000)-N(2000) average')

print('QC passed: complete SON years and SON(2000) month-membership rule.')


In [ ]:
clim = son.sel(year=slice(CLIM_START, CLIM_END)).mean('year')

# Baseline integrity check: must be exactly 30 SON seasons.
clim_years = son.sel(year=slice(CLIM_START, CLIM_END))['year'].values.astype(int)
if len(clim_years) != 30:
    raise AssertionError(f'Climatology year count is {len(clim_years)}, expected 30 for 1991-2020')

anom = son - clim
anom.name = 'precipitation_anomaly'
anom.attrs['description'] = 'SON precipitation anomaly relative to 1991-2020 climatology'

print('Anomaly cube shape (year, lat, lon):', anom.shape)
print('Climatology years:', clim_years[0], 'to', clim_years[-1], f'(n={len(clim_years)})')


In [ ]:
def apply_sign_convention(components: np.ndarray, scores: np.ndarray):
    """Flip mode signs so the largest-absolute loading is positive."""
    comp = components.copy()
    pcs = scores.copy()
    for m in range(comp.shape[0]):
        k = int(np.nanargmax(np.abs(comp[m, :])))
        if comp[m, k] < 0:
            comp[m, :] *= -1.0
            pcs[:, m] *= -1.0
    return comp, pcs

ny, nlat, nlon = anom.shape
X = anom.values.reshape(ny, nlat * nlon).astype(np.float64)

# Keep columns that are finite for all years (removes all-NaN and any partial-NaN columns).
valid_mask = np.all(np.isfinite(X), axis=0)
X_valid = X[:, valid_mask]

X_centered = X_valid - X_valid.mean(axis=0, keepdims=True)
if not np.isfinite(X_centered).all():
    raise AssertionError('Non-finite values remain in PCA input matrix')

n_modes = int(min(EOF_NMODES, X_centered.shape[0], X_centered.shape[1]))
pca = PCA(n_components=n_modes, svd_solver='full')
pcs = pca.fit_transform(X_centered)
components = pca.components_

# Multiply EOF and PC by -1 as requested
# components = -components
# pcs = -pcs

components, pcs = apply_sign_convention(components, pcs)

# Restore EOF maps to full lat-lon grid.
eof_flat_full = np.full((n_modes, nlat * nlon), np.nan, dtype=np.float64)
eof_flat_full[:, valid_mask] = components
eof_maps = eof_flat_full.reshape(n_modes, nlat, nlon)

eof_da = xr.DataArray(
    eof_maps,
    dims=('mode', 'lat', 'lon'),
    coords={'mode': np.arange(1, n_modes + 1), 'lat': anom['lat'], 'lon': anom['lon']},
    name='eof',
)

ev_ratio = pca.explained_variance_ratio_
ev_cum = np.cumsum(ev_ratio)

# Matrix sanity checks.
if pcs.shape[0] != ny:
    raise AssertionError(f'PC sample count mismatch: {pcs.shape[0]} vs {ny}')
if np.any(ev_ratio < 0):
    raise AssertionError('Explained variance ratio has negative values')
if np.any(np.diff(ev_cum) < -1e-12):
    raise AssertionError('Cumulative explained variance is not monotonic non-decreasing')

print('PCA input matrix shape:', X_centered.shape)
print('Valid grid points used:', X_centered.shape[1], 'out of', nlat * nlon)
print('Computed modes:', n_modes)
for i, v in enumerate(ev_ratio, start=1):
    print(f'  EOF{i}: {v * 100:.2f}%')


In [ ]:
year_index = pd.Index(anom['year'].values.astype(int), name='year')

var_df = pd.DataFrame({
    'mode': np.arange(1, n_modes + 1, dtype=int),
    'explained_variance_ratio': ev_ratio,
    'explained_variance_percent': ev_ratio * 100.0,
    'cumulative_percent': ev_cum * 100.0,
})

pc_df = pd.DataFrame({'year': year_index.values.astype(int)})
for i in range(n_modes):
    pc_df[f'PC{i + 1}'] = pcs[:, i]

var_name = f'mswep_son_eof_variance_{year_index.min()}_{year_index.max()}_base{CLIM_START}_{CLIM_END}.csv'
pc_name = f'mswep_son_pc_scores_{year_index.min()}_{year_index.max()}_base{CLIM_START}_{CLIM_END}.csv'

var_csv = OUT_TABLES / var_name
pc_csv = OUT_TABLES / pc_name

var_df.to_csv(var_csv, index=False)
pc_df.to_csv(pc_csv, index=False)

print('Saved variance table:', var_csv)
print('Saved PC table:', pc_csv)
var_df


In [ ]:
# EOF1 map with Cartopy
eof1 = eof_da.sel(mode=1)
eof1 = -eof1
vmax = float(np.nanmax(np.abs(eof1.values)))
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1e-12

proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(8.5, 5.5), constrained_layout=True)
ax = plt.axes(projection=proj)

im = ax.pcolormesh(
    eof1['lon'],
    eof1['lat'],
    eof1,
    shading='auto',
    cmap='RdBu_r',
    vmin=-vmax,
    vmax=vmax,
    transform=ccrs.PlateCarree(),
)

ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
ax.coastlines(resolution='10m', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)

xticks = np.arange(LON_MIN, LON_MAX + 1, 10)
yticks = np.arange(LAT_MIN, LAT_MAX + 1, 5)
ax.set_xticks(xticks, crs=ccrs.PlateCarree())
ax.set_yticks(yticks, crs=ccrs.PlateCarree())
ax.xaxis.set_major_formatter(LongitudeFormatter())
ax.yaxis.set_major_formatter(LatitudeFormatter())
ax.gridlines(draw_labels=False, linewidth=0.4, color='gray', alpha=0.4, linestyle='--')

ax.set_title(f'MSWEP SON EOF1 ({ev_ratio[0] * 100:.1f}%)')
cb = fig.colorbar(im, ax=ax, pad=0.03, shrink=0.9)
cb.set_label('Loading')

map_png = OUT_PNG / 'mswep_son_eof1_map_cartopy.png'
fig.savefig(map_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', map_png)


In [ ]:
# PC time series
n_plot = min(N_PLOT_MODES, n_modes)
fig, axes = plt.subplots(n_plot, 1, figsize=(11, 2.8 * n_plot), sharex=True, constrained_layout=True)
if n_plot == 1:
    axes = [axes]

x = year_index.values
for i in range(n_plot):
    ax = axes[i]
    y = pcs[:, i]
    ax.plot(x, y, color='tab:blue', lw=1.7)
    ax.axhline(0.0, color='0.3', lw=0.9, ls='--')
    ax.set_ylabel(f'PC{i + 1}')
    ax.set_title(f'PC{i + 1} ({ev_ratio[i] * 100:.1f}%)')

axes[-1].set_xlabel('SON year')
fig.suptitle('MSWEP SON Principal Components', y=1.02)
pc_png = OUT_PNG / 'mswep_son_pc_timeseries_1to3.png'
fig.savefig(pc_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', pc_png)


In [ ]:
# PC1 vs Nino3.4 (SON mean) correlation
nino_df = pd.read_csv(NINO34_PATH)
nino_df.columns = [c.strip() for c in nino_df.columns]

date_col = nino_df.columns[0]
value_col = nino_df.columns[1]
nino_df[date_col] = pd.to_datetime(nino_df[date_col], errors='coerce')
nino_df[value_col] = pd.to_numeric(nino_df[value_col], errors='coerce')
nino_df.loc[nino_df[value_col] <= -999, value_col] = np.nan
nino_df = nino_df.dropna(subset=[date_col, value_col]).copy()

nino_monthly = xr.DataArray(
    nino_df[value_col].values,
    dims=('time',),
    coords={'time': pd.DatetimeIndex(nino_df[date_col].values)},
    name='nino34',
)

# Use SON instead of DJF
def build_complete_son(da: xr.DataArray) -> xr.DataArray:
    month = da['time'].dt.month
    year = da['time'].dt.year
    is_son = month.isin([9, 10, 11])
    da_son = da.sel(time=is_son)
    son_year = da_son['time'].dt.year
    da_son = da_son.assign_coords(son_year=('time', son_year.data))
    month_count = da_son['time'].groupby('son_year').count()
    full_years = month_count['son_year'].where(month_count == 3, drop=True).values.astype(int)
    son = da_son.groupby('son_year').mean('time').sel(son_year=full_years)
    son = son.rename({'son_year': 'year'}).sortby('year')
    return son

nino_son = build_complete_son(nino_monthly).to_pandas()
nino_son.name = 'nino34_son'

pc1 = pd.Series(pcs[:, 0], index=year_index.astype(int), name='PC1')
pc1_nino = pd.concat([pc1, nino_son], axis=1).dropna()

# Multiply PC1 by -1 as requested
pc1_nino['PC1'] = -pc1_nino['PC1']

if len(pc1_nino) < 3:
    raise AssertionError('Insufficient overlapping SON years between PC1 and Nino3.4 for correlation')

r_total = float(pc1_nino['PC1'].corr(pc1_nino['nino34_son']))

corr_df = pc1_nino.reset_index().rename(columns={'index': 'year'})
corr_csv = OUT_TABLES / f'mswep_pc1_nino34_son_{corr_df.year.min()}_{corr_df.year.max()}.csv'
corr_df.to_csv(corr_csv, index=False)

pc1_z = (pc1_nino['PC1'] - pc1_nino['PC1'].mean()) / pc1_nino['PC1'].std(ddof=0)
nino_z = (pc1_nino['nino34_son'] - pc1_nino['nino34_son'].mean()) / pc1_nino['nino34_son'].std(ddof=0)

fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax.plot(pc1_nino.index, pc1_z, color='tab:blue', lw=1.8, label='PC1 (normalized)')
ax.plot(pc1_nino.index, nino_z, color='tab:red', lw=1.8, label='Nino3.4 SON (normalized)')
ax.axhline(0, color='0.3', lw=0.9, ls='--')
ax.set_xlabel('SON year')
ax.set_ylabel('Standardized value')
ax.set_title(f'PC1 vs Nino3.4 SON mean | r = {r_total:.3f}')
ax.legend(loc='upper left')

corr_png = OUT_PNG / 'mswep_pc1_vs_nino34_son_correlation.png'
fig.savefig(corr_png, dpi=180, bbox_inches='tight')
plt.show()

print(f'Total correlation (PC1 vs Nino3.4 SON): r = {r_total:.4f}')
print('Saved:', corr_png)
print('Saved:', corr_csv)


In [ ]:
# PC1 vs Nino3.4 SON mean (raw, twin y-axis)
fig, ax1 = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax2 = ax1.twinx()

ax1.plot(pc1_nino.index, pc1_nino['PC1'], color='tab:blue', lw=1.8, label='PC1')
ax2.plot(pc1_nino.index, pc1_nino['nino34_son'], color='tab:red', lw=1.8, label='Nino3.4 SON')

ax1.set_xlabel('SON year')
ax1.set_ylabel('PC1', color='tab:blue')
ax2.set_ylabel('Nino3.4 SON', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax1.axhline(0, color='0.3', lw=0.9, ls='--')
ax2.axhline(0, color='0.3', lw=0.9, ls='--')
fig.suptitle(f'PC1 vs Nino3.4 SON | r = {r_total:.3f}')

corr_raw_png = OUT_PNG / 'mswep_pc1_vs_nino34_son_raw_twin.png'
fig.savefig(corr_raw_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', corr_raw_png)


In [ ]:
# PC1 vs DMI (DJF mean) correlation (raw, twin y-axis)
dmi_raw = dmi_son  # already DJF mean, raw values
pc1_dmi = pd.concat([pc1, dmi_raw], axis=1).dropna()
pc1_dmi['PC1'] = -pc1_dmi['PC1']  # sign convention as before

if len(pc1_dmi) < 3:
    raise AssertionError('Insufficient overlapping DJF years between PC1 and DMI for correlation')

r_pc1_dmi = float(pc1_dmi['PC1'].corr(pc1_dmi['dmi_djf']))

fig, ax1 = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax2 = ax1.twinx()

ax1.plot(pc1_dmi.index, pc1_dmi['PC1'], color='tab:blue', lw=1.8, label='PC1')
ax2.plot(pc1_dmi.index, pc1_dmi['dmi_djf'], color='tab:orange', lw=1.8, label='DMI DJF')

ax1.set_xlabel('DJF year')
ax1.set_ylabel('PC1 (raw)', color='tab:blue')
ax2.set_ylabel('DMI DJF (raw)', color='tab:orange')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax2.tick_params(axis='y', labelcolor='tab:orange')
ax1.axhline(0, color='0.3', lw=0.9, ls='--')
ax2.axhline(0, color='0.3', lw=0.9, ls='--')
fig.suptitle(f'PC1 vs DMI DJF | r = {r_pc1_dmi:.3f}')

corr_pc1_dmi_raw_png = OUT_PNG / 'mswep_pc1_vs_dmi_djf_raw_twin.png'
fig.savefig(corr_pc1_dmi_raw_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', corr_pc1_dmi_raw_png)

In [ ]:
# PC2 vs Nino3.4 (SON mean) correlation
pc2 = pd.Series(pcs[:, 1], index=year_index.astype(int), name='PC2')
pc2_nino = pd.concat([pc2, nino_son], axis=1).dropna()

# Multiply PC2 by -1 as requested
pc2_nino['PC2'] = -pc2_nino['PC2']

if len(pc2_nino) < 3:
    raise AssertionError('Insufficient overlapping SON years between PC2 and Nino3.4 for correlation')

r_total2 = float(pc2_nino['PC2'].corr(pc2_nino['nino34_son']))

corr2_df = pc2_nino.reset_index().rename(columns={'index': 'year'})
corr2_csv = OUT_TABLES / f'mswep_pc2_nino34_son_{corr2_df.year.min()}_{corr2_df.year.max()}.csv'
corr2_df.to_csv(corr2_csv, index=False)

pc2_z = (pc2_nino['PC2'] - pc2_nino['PC2'].mean()) / pc2_nino['PC2'].std(ddof=0)
nino_z2 = (pc2_nino['nino34_son'] - pc2_nino['nino34_son'].mean()) / pc2_nino['nino34_son'].std(ddof=0)

fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax.plot(pc2_nino.index, pc2_z, color='tab:green', lw=1.8, label='PC2 (z-score)')
ax.plot(pc2_nino.index, nino_z2, color='tab:red', lw=1.8, label='Nino3.4 SON (z-score)')
ax.axhline(0, color='0.3', lw=0.9, ls='--')
ax.set_xlabel('SON year')
ax.set_ylabel('Standardized value')
ax.set_title(f'PC2 vs Nino3.4 SON mean | r = {r_total2:.3f}')
ax.legend(loc='upper left')

corr2_png = OUT_PNG / 'mswep_pc2_vs_nino34_son_correlation.png'
fig.savefig(corr2_png, dpi=180, bbox_inches='tight')
plt.show()

print(f'Total correlation (PC2 vs Nino3.4 SON): r = {r_total2:.4f}')
print('Saved:', corr2_png)
print('Saved:', corr2_csv)


In [ ]:
# # PC2 vs DMI (DJF mean) correlation
# dmi_df = pd.read_csv('/Users/rizzie/Academic/9_TugasAkhir/data/index/dmi.had.long.csv')
# dmi_df.columns = [c.strip() for c in dmi_df.columns]

# dmi_date_col = dmi_df.columns[0]
# dmi_value_col = dmi_df.columns[1]
# dmi_df[dmi_date_col] = pd.to_datetime(dmi_df[dmi_date_col], errors='coerce')
# dmi_df[dmi_value_col] = pd.to_numeric(dmi_df[dmi_value_col], errors='coerce')
# dmi_df.loc[dmi_df[dmi_value_col] <= -999, dmi_value_col] = np.nan
# dmi_df = dmi_df.dropna(subset=[dmi_date_col, dmi_value_col]).copy()

# dmi_monthly = xr.DataArray(
#     dmi_df[dmi_value_col].values,
#     dims=('time',),
#     coords={'time': pd.DatetimeIndex(dmi_df[dmi_date_col].values)},
#     name='dmi',
# )

# dmi_djf = build_complete_djf(dmi_monthly).to_pandas()
# dmi_djf.name = 'dmi_djf'

# pc2_dmi = pd.concat([pc2, dmi_djf], axis=1).dropna()
# pc2_dmi['PC2'] = -pc2_dmi['PC2']

# if len(pc2_dmi) < 3:
#     raise AssertionError('Insufficient overlapping DJF years between PC2 and DMI for correlation')

# r_pc2_dmi = float(pc2_dmi['PC2'].corr(pc2_dmi['dmi_djf']))

# corr_dmi_df = pc2_dmi.reset_index().rename(columns={'index': 'year'})
# corr_dmi_csv = OUT_TABLES / f'mswep_pc2_dmi_djf_{corr_dmi_df.year.min()}_{corr_dmi_df.year.max()}.csv'
# corr_dmi_df.to_csv(corr_dmi_csv, index=False)

# pc2_z_dmi = (pc2_dmi['PC2'] - pc2_dmi['PC2'].mean()) / pc2_dmi['PC2'].std(ddof=0)
# dmi_z = (pc2_dmi['dmi_djf'] - pc2_dmi['dmi_djf'].mean()) / pc2_dmi['dmi_djf'].std(ddof=0)

# fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
# ax.plot(pc2_dmi.index, pc2_z_dmi, color='tab:green', lw=1.8, label='PC2 (z-score)')
# ax.plot(pc2_dmi.index, dmi_z, color='tab:orange', lw=1.8, label='DMI DJF (z-score)')
# ax.axhline(0, color='0.3', lw=0.9, ls='--')
# ax.set_xlabel('DJF year')
# ax.set_ylabel('Standardized value')
# ax.set_title(f'PC2 vs DMI DJF mean | r = {r_pc2_dmi:.3f}')
# ax.legend(loc='upper left')

# corr_dmi_png = OUT_PNG / 'mswep_pc2_vs_dmi_djf_correlation.png'
# fig.savefig(corr_dmi_png, dpi=180, bbox_inches='tight')
# plt.show()

# print(f'Total correlation (PC2 vs DMI DJF): r = {r_pc2_dmi:.4f}')
# print('Saved:', corr_dmi_png)
# print('Saved:', corr_dmi_csv)

In [ ]:
# PC2 vs Nino3.4 SON mean (raw, twin y-axis)
fig, ax1 = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax2 = ax1.twinx()

ax1.plot(pc2_nino.index, pc2_nino['PC2'], color='tab:green', lw=1.8, label='PC2')
ax2.plot(pc2_nino.index, pc2_nino['nino34_son'], color='tab:red', lw=1.8, label='Nino3.4 SON')

ax1.set_xlabel('SON year')
ax1.set_ylabel('PC2 (raw)', color='tab:green')
ax2.set_ylabel('Nino3.4 SON (raw)', color='tab:red')
ax1.tick_params(axis='y', labelcolor='tab:green')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax1.axhline(0, color='0.3', lw=0.9, ls='--')
ax2.axhline(0, color='0.3', lw=0.9, ls='--')
fig.suptitle(f'PC2 vs Nino3.4 SON | r = {r_total2:.3f}')

corr2_raw_png = OUT_PNG / 'mswep_pc2_vs_nino34_son_raw_twin.png'
fig.savefig(corr2_raw_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', corr2_raw_png)


In [ ]:
# # Replicate PC2 vs DMI DJF mean (raw, twin y-axis, no normalization)
# fig, ax1 = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
# ax2 = ax1.twinx()

# ax1.plot(pc2_dmi.index, pc2_dmi['PC2'], color='tab:green', lw=1.8, label='PC2')
# ax2.plot(pc2_dmi.index, pc2_dmi['dmi_djf'], color='tab:orange', lw=1.8, label='DMI DJF')

# ax1.set_xlabel('DJF year')
# ax1.set_ylabel('PC2 (raw)', color='tab:green')
# ax2.set_ylabel('DMI DJF (raw)', color='tab:orange')
# ax1.tick_params(axis='y', labelcolor='tab:green')
# ax2.tick_params(axis='y', labelcolor='tab:orange')
# ax1.axhline(0, color='0.3', lw=0.9, ls='--')
# ax2.axhline(0, color='0.3', lw=0.9, ls='--')
# fig.suptitle(f'PC2 vs DMI DJF | r = {r_pc2_dmi:.3f}')

# corr_dmi_raw_png = OUT_PNG / 'mswep_pc2_vs_dmi_djf_raw_twin.png'
# fig.savefig(corr_dmi_raw_png, dpi=180, bbox_inches='tight')
# plt.show()
# print('Saved:', corr_dmi_raw_png)

In [ ]:
# Scree + cumulative variance
fig, ax1 = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)
mode_nums = np.arange(1, n_modes + 1)

ax1.bar(mode_nums, ev_ratio * 100.0, color='tab:blue', alpha=0.75, label='Explained variance (%)')
ax1.set_xlabel('EOF mode')
ax1.set_ylabel('Explained variance (%)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.set_xticks(mode_nums)

ax2 = ax1.twinx()
ax2.plot(mode_nums, ev_cum * 100.0, color='tab:red', marker='o', lw=1.8, label='Cumulative (%)')
ax2.set_ylabel('Cumulative variance (%)', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax2.set_ylim(0, 100)

fig.suptitle('MSWEP SON EOF Scree Plot')
scree_png = OUT_PNG / 'mswep_son_eof_scree.png'
fig.savefig(scree_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', scree_png)


In [ ]:
# Save EOF fields to NetCDF
year_min = int(year_index.min())
year_max = int(year_index.max())

out_nc = OUT_BASE / f'mswep_son_eof_maps_{year_min}_{year_max}_base{CLIM_START}_{CLIM_END}.nc'

ds_out = xr.Dataset(
    data_vars={
        'eof': eof_da,
        'explained_variance_ratio': xr.DataArray(ev_ratio, dims=('mode',), coords={'mode': eof_da['mode']}),
    },
    attrs={
        'title': 'MSWEP Indonesia SON EOF/PCA',
        'source_file': str(DATA_PATH),
        'domain_lat_min': LAT_MIN,
        'domain_lat_max': LAT_MAX,
        'domain_lon_min': LON_MIN,
        'domain_lon_max': LON_MAX,
        'son_definition': 'S(y), O(y), N(y)',
        'climatology_base_period': f'{CLIM_START}-{CLIM_END}',
        'method': 'Covariance-based PCA on SON anomaly field, no latitude weighting, no pointwise standardization',
    },
)

# Output consistency checks.
if ds_out['eof'].shape[1:] != (len(anom['lat']), len(anom['lon'])):
    raise AssertionError('EOF grid shape mismatch with anomaly grid')
if len(pc_df) != len(year_index):
    raise AssertionError('PC table row count mismatch with SON years')

ds_out.to_netcdf(out_nc)
print('Saved NetCDF:', out_nc)


In [ ]:
# Reproducibility check: rerun PCA on the same centered matrix and compare after sign convention.
pca2 = PCA(n_components=n_modes, svd_solver='full')
pcs2 = pca2.fit_transform(X_centered)
comp2 = pca2.components_
comp2, pcs2 = apply_sign_convention(comp2, pcs2)

if not np.allclose(comp2, components, atol=1e-10, rtol=1e-10):
    raise AssertionError('Reproducibility check failed: EOF components changed across identical rerun')
if not np.allclose(pcs2, pcs, atol=1e-10, rtol=1e-10):
    raise AssertionError('Reproducibility check failed: PCs changed across identical rerun')

print('All checks passed, including reproducibility.')


## Running Correlation: Raw PC1 vs Raw Nino3.4 DJF (15-year window)

This analysis computes the running (moving window) correlation between the raw PC1 time series and the raw Nino3.4 DJF mean index using a 15-year window. The plot below shows the correlation coefficient for each window, centered on the window's midpoint year.

In [ ]:
# Compute running correlation (15-year window) between raw PC1 and raw Nino3.4 SON mean
window = 15
pc1_raw = pc1_nino['PC1']  # already raw, not normalized
nino_raw = pc1_nino['nino34_son']  # already raw, not normalized
years_common = pc1_nino.index.astype(int)
n = len(years_common)
corrs = []
mid_years = []
for i in range(n - window + 1):
    pc1_win = pc1_raw.iloc[i:i+window]
    nino_win = nino_raw.iloc[i:i+window]
    if pc1_win.isnull().any() or nino_win.isnull().any():
        corrs.append(np.nan)
    else:
        corrs.append(pc1_win.corr(nino_win))
    mid_years.append(years_common[i + window // 2])
corrs = np.array(corrs)
mid_years = np.array(mid_years)

fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax.plot(mid_years, corrs, color='tab:purple', lw=2.2, marker='o')
ax.axhline(0, color='0.3', lw=0.9, ls='--')
ax.set_xlabel('Midpoint SON year')
ax.set_ylabel('Running correlation (r)')
ax.set_title('Running correlation (15-year window): Raw PC1 vs Raw Nino3.4 SON')
ax.grid(True, linestyle='--', alpha=0.5)

run_corr_png = OUT_PNG / 'mswep_pc1_nino34_son_running_corr_15yr.png'
fig.savefig(run_corr_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', run_corr_png)


In [ ]:
# Running regression slope (PC1 ~ Nino3.4 SON) using moving window
win = window  # uses existing window (currently 15)

pc1_raw = pc1_nino['PC1']
nino_raw = pc1_nino['nino34_son']
years_common = pc1_nino.index.astype(int)

if len(years_common) < win:
    raise AssertionError(f'Not enough data for a {win}-year running regression')

slopes = []
mid_years = []

for i in range(len(years_common) - win + 1):
    x = nino_raw.iloc[i:i + win].to_numpy(dtype=float)
    y = pc1_raw.iloc[i:i + win].to_numpy(dtype=float)

    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        slopes.append(np.nan)
    else:
        # OLS slope for y = a + b*x
        b = np.polyfit(x[m], y[m], 1)[0]
        slopes.append(b)

    mid_years.append(years_common[i + win // 2])

slopes = np.array(slopes, dtype=float)
mid_years = np.array(mid_years, dtype=int)

fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax.plot(mid_years, slopes, color='tab:purple', lw=2.2, marker='o')
ax.axhline(0, color='0.3', lw=0.9, ls='--')
ax.set_xlabel('Midpoint SON year')
ax.set_ylabel('Running regression slope (PC1 per Nino3.4 unit)')
ax.set_title(f'Running regression slope ({win}-year): PC1 vs Nino3.4 SON')
ax.grid(True, linestyle='--', alpha=0.5)

run_slope_png = OUT_PNG / f'mswep_pc1_nino34_son_running_regression_slope_{win}yr.png'
fig.savefig(run_slope_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', run_slope_png)


In [ ]:
# NumPy compatibility for older libraries (e.g., pycwt) still using deprecated aliases
if not hasattr(np, 'int'):
	np.int = int

import pycwt as wavelet

# Prepare time series from common years
t_years = pc1_nino.index.to_numpy(dtype=float)
dt = 1  # 1 year interval

# Remove mean and normalize
pc1_vals = pc1_nino['PC1'].to_numpy(dtype=float)
nino_vals = pc1_nino['nino34_son'].to_numpy(dtype=float)
pc1_norm = (pc1_vals - pc1_vals.mean()) / pc1_vals.std(ddof=0)
nino_norm = (nino_vals - nino_vals.mean()) / nino_vals.std(ddof=0)

# Compute wavelet coherence
# This build returns: WCT, phase, coi, freq, signif
WCT, phase, coi, freq, signif = wavelet.wct(pc1_norm, nino_norm, dt)

# Plot manually (replacement for missing wavelet.plot_wct)
period = 1.0 / freq
T, P = np.meshgrid(t_years, period)

fig, ax = plt.subplots(figsize=(11, 5), constrained_layout=True)
levels = np.linspace(0, 1, 21)
cf = ax.contourf(T, P, np.abs(WCT), levels=levels, cmap='viridis', extend='max')

# 95% significance contour (if available and compatible)
if signif is not None and np.ndim(signif) == 1 and len(signif) == len(period):
	sig2d = signif[:, None] * np.ones_like(WCT)
	ax.contour(T, P, np.abs(WCT), levels=sig2d[:, 0:1].mean(axis=1), colors='none')
	ax.contour(T, P, np.abs(WCT) - sig2d, levels=[0], colors='white', linewidths=1.0)

# Cone of influence (if available)
if coi is not None and len(coi) == len(t_years):
	ax.plot(t_years, coi, color='k', lw=1.2, ls='--')
	ax.fill_between(t_years, coi, period.max(), color='white', alpha=0.35)

ax.set_yscale('log', base=2)
ax.set_ylim(period.min(), period.max())
ax.set_xlabel('SON year')
ax.set_ylabel('Period (years)')
ax.set_title('Wavelet Coherence: PC1 vs Nino3.4 SON')

cb = fig.colorbar(cf, ax=ax, pad=0.02)
cb.set_label('Wavelet coherence')

plt.show()


## Running EOF Analysis (Moving Window)

This section demonstrates how to perform a running EOF (PCA) analysis on DJF precipitation anomalies using a moving window approach. For each window, the leading EOF spatial pattern and explained variance are recalculated, allowing us to track changes in the dominant mode of variability over time.

In [ ]:
# Running EOF (PCA) analysis with a moving window (15 years)
window = 15
ny, nlat, nlon = anom.shape
years_all = anom['year'].values.astype(int)
eof1_maps = []
eof1_years = []
eof1_var = []
for i in range(ny - window + 1):
    anom_win = anom.isel(year=slice(i, i+window))
    X_win = anom_win.values.reshape(window, nlat * nlon)
    valid_mask_win = np.all(np.isfinite(X_win), axis=0)
    X_valid_win = X_win[:, valid_mask_win]
    X_centered_win = X_valid_win - X_valid_win.mean(axis=0, keepdims=True)
    pca_win = PCA(n_components=1, svd_solver='full')
    pcs_win = pca_win.fit_transform(X_centered_win)
    comp_win = pca_win.components_[0]
    # Apply sign convention
    k_win = int(np.nanargmax(np.abs(comp_win)))
    if comp_win[k_win] < 0:
        comp_win *= -1.0
        pcs_win[:, 0] *= -1.0
    # Restore to full grid
    eof_flat_full_win = np.full(nlat * nlon, np.nan, dtype=np.float64)
    eof_flat_full_win[valid_mask_win] = comp_win
    eof_map_win = eof_flat_full_win.reshape(nlat, nlon)
    eof1_maps.append(eof_map_win)
    eof1_years.append(years_all[i + window // 2])
    eof1_var.append(pca_win.explained_variance_ratio_[0])
eof1_maps = np.array(eof1_maps)
eof1_years = np.array(eof1_years)
eof1_var = np.array(eof1_var)

# Plot running EOF1 explained variance
fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)
ax.plot(eof1_years, eof1_var * 100, color='tab:blue', lw=2.2, marker='o')
ax.set_xlabel('Midpoint DJF year')
ax.set_ylabel('EOF1 explained variance (%)')
ax.set_title('Running EOF1 explained variance (15-year window)')
ax.grid(True, linestyle='--', alpha=0.5)

run_eof_var_png = OUT_PNG / 'mswep_running_eof1_explained_variance_15yr.png'
fig.savefig(run_eof_var_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', run_eof_var_png)

# Optionally: plot a few example EOF1 maps for selected windows
sel_idxs = [0, len(eof1_years)//2, -1]
for idx in sel_idxs:
    fig, ax = plt.subplots(figsize=(8.5, 5.5), constrained_layout=True)
    eof_map = eof1_maps[idx]
    vmax = float(np.nanmax(np.abs(eof_map)))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1e-12
    im = ax.pcolormesh(anom['lon'], anom['lat'], eof_map, shading='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'EOF1 map (window centered {eof1_years[idx]})')
    cb = fig.colorbar(im, ax=ax, pad=0.03, shrink=0.9)
    cb.set_label('Loading')
    map_png = OUT_PNG / f'mswep_running_eof1_map_{eof1_years[idx]}.png'
    fig.savefig(map_png, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved:', map_png)

In [ ]:
# Plot running PC1 time series (window midpoints)
fig, ax = plt.subplots(figsize=(11, 4.8), constrained_layout=True)

mean_pc1 = []
for i, mid_year in enumerate(eof1_years):
    anom_win = anom.isel(year=slice(i, i + window))
    X_win = anom_win.values.reshape(window, nlat * nlon)

    valid_mask_win = np.all(np.isfinite(X_win), axis=0)
    X_valid_win = X_win[:, valid_mask_win]
    X_centered_win = X_valid_win - X_valid_win.mean(axis=0, keepdims=True)

    pca_win_local = PCA(n_components=1, svd_solver='full')
    pcs_win_local = pca_win_local.fit_transform(X_centered_win)[:, 0]
    comp_win_local = pca_win_local.components_[0]

    # Sign convention (same as running EOF cell)
    k_win = int(np.nanargmax(np.abs(comp_win_local)))
    if comp_win_local[k_win] < 0:
        pcs_win_local *= -1.0

    win_years = np.arange(mid_year - window // 2, mid_year + window // 2 + 1)
    ax.plot(win_years, pcs_win_local, color='tab:blue', alpha=0.2, lw=1.1)
    mean_pc1.append(float(np.mean(pcs_win_local)))

# Midpoint summary
ax.plot(eof1_years, mean_pc1, color='tab:orange', lw=2.2, marker='o', label='Mean PC1 (window)')
ax.axhline(0, color='0.3', lw=0.9, ls='--')
ax.set_xlabel('Midpoint DJF year')
ax.set_ylabel('PC1 (window mean)')
ax.set_title('Running PC1 (mean per 15-year window)')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend()

run_pc1_png = OUT_PNG / 'mswep_running_pc1_mean_15yr.png'
fig.savefig(run_pc1_png, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', run_pc1_png)

In [ ]:
fig = plt.figure(figsize=(8.5, 5.5), constrained_layout=True)
ax = plt.axes(projection=proj)

# Draw the study region as a pink rectangle
region_lon = [70, 90, 90, 70, 70]
region_lat = [5, 5, 30, 30, 5]
ax.plot(region_lon, region_lat, color='hotpink', linewidth=2.5, transform=ccrs.PlateCarree())
ax.fill(region_lon, region_lat, color='pink', alpha=0.4, transform=ccrs.PlateCarree())

ax.set_extent([65, 95, 0, 35], crs=ccrs.PlateCarree())
ax.coastlines(resolution='10m', linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)

ax.set_xticks(np.arange(70, 91, 5), crs=ccrs.PlateCarree())
ax.set_yticks(np.arange(5, 31, 5), crs=ccrs.PlateCarree())
ax.xaxis.set_major_formatter(LongitudeFormatter())
ax.yaxis.set_major_formatter(LatitudeFormatter())
ax.gridlines(draw_labels=False, linewidth=0.4, color='gray', alpha=0.4, linestyle='--')

ax.set_title('Study Region: 70°–90°E, 5°–30°N (pink)')
plt.show()
